In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_seq_items', None)
pd.set_option('display.max_columns', None)

from matplotlib import pyplot as plt
import re, os, csv, json
from datetime import datetime

import altair as alt
alt.data_transformers.disable_max_rows()

In [ ]:
d = lambda x: datetime.strptime(x, '%Y')

df = pd.read_csv("World Energy Consumption.csv", parse_dates=['year'], date_parser=d)
df

In [ ]:
print("Data shape\n", '-'*50, '\n', str(df.shape))

print("\nColumns\n", '-'*50, '\n', str(df.columns))

print("\nDatatypes\n", '-'*50, '\n', str(df.dtypes))

print("\nNumber of null or NaN columns\n", '-'*50, '\n', df.isna().sum())

print("\nPercentage of nulls columns\n", '-'*50, '\n', 100*df.isna().sum()/len(df))

print("\nNumber of Unique Values\n", '-'*50, '\n', str(df.nunique()))

In [ ]:
# drop_na_rows_importantCols = ['iso_code', 'population', 'gdp']
# df = df.dropna(subset = drop_na_rows_importantCols)
# df

In [ ]:
df.describe()

In [ ]:
df["gdp_per_capita"] = df["gdp"] / df["population"]
df["electricity_demand_per_capita"] = df["electricity_demand"] / df["population"]

df

# Plot 1
- scatter plot
- GDP per Capita on X
- total energy demand per capita on y
- Most recent year

In [ ]:
df_2018 = df[df['year'] == '2018-01-01']
df_2018

In [ ]:

scatter1 = alt.Chart(df_2018).mark_circle(size=60).encode(
    x=alt.X('gdp_per_capita:Q', title='GDP per Capita'),
    y=alt.Y('energy_per_capita:Q', title='Energy Demand per Capita (kWh/person)'),
    color=alt.Color('low_carbon_share_energy', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
    tooltip=['country', 'gdp_per_capita', 'energy_per_gdp', 'low_carbon_share_energy']
).properties(
    # title='GDP per Capita vs Energy Intensity (2018)',
    width=600,
    height=400
)

scatter1

In [ ]:

scatter1 = alt.Chart(df_2018).mark_circle(size=60).encode(
    x=alt.X('gdp_per_capita:Q', title='GDP per Capita'),
    y=alt.Y('electricity_demand_per_capita:Q', title='Energy Demand per Capita (kWh/person)'),
    color=alt.Color('low_carbon_share_elec', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
    tooltip=['country', 'gdp_per_capita', 'energy_per_gdp', 'low_carbon_share_elec']
).properties(
    # title='GDP per Capita vs Energy Intensity (2018)',
    width=600,
    height=400
)

scatter1

In [ ]:
scatter2 = alt.Chart(df_2018).mark_circle(size=60).encode(
    x=alt.X('gdp_per_capita:Q', title='GDP per Capita',scale=alt.Scale(type='log')),
    y=alt.Y('energy_per_capita:Q', title='Energy Demand per Capita (kWh/person)', scale=alt.Scale(type='log')),
    color=alt.Color('low_carbon_share_energy', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
    tooltip=['country', 'gdp_per_capita', 'energy_per_gdp', 'low_carbon_share_energy']
).properties(
    title='Energy Demand as a function of GDP in 2018',
    width=600,
    height=400
)

scatter2

# Plot 2
- time series plot
- World
- x = year
- y = total energy demand, energy from low carbon sources (%)

In [ ]:
df_world = df[df['country'] == 'World']
df_world = df_world[df_world['year']>='1965']
df_world

In [ ]:
# alt.Chart(df_world).mark_area().encode(
#     x="year",
#     y=alt.Y("energy_per_capita:Q"),
#     y2=alt.Y2("low_carbon_energy_per_capita:Q"),
#     tooltip=['year', 'energy_per_capita']
#     # color="source:N"
# )

In [ ]:
alt.Chart(df_world).mark_area().encode(
    x="year",
    y=alt.Y("energy_per_capita:Q"),
    y2=alt.Y2("low_carbon_energy_per_capita:Q"),
    tooltip=['year', 'energy_per_capita']
    # color="source:N"
)

base = alt.Chart(df_world).mark_area().encode(x="year", tooltip=['year', 'energy_per_capita','low_carbon_energy_per_capita'])

alt.layer(
    base.mark_area().encode(y=alt.Y("energy_per_capita:Q", title='total')),
    base.mark_area(color='red').encode(y=alt.Y("low_carbon_energy_per_capita:Q", title='green'))
)

In [ ]:
alt.Chart(df_world).transform_fold(
    fold=['energy_per_capita', 'low_carbon_energy_per_capita']
    # as_=['']
).mark_area().encode(
    x=alt.X('year:T', title='Year'),
    y=alt.Y('value:Q', title='Energy Consumption per Capita (kWh/person)'),
    color=alt.Color('key:N', legend=alt.Legend(title='Energy Source'))
).properties(
    title='World Energy Consumption Since 1965',
    width=600,
    height=400
)
# ).transform_calculate(
#     # Define the custom labels based on the original 'variable' value
#     custom_label="datum.variable === 'energy_per_capita' ? 'Total' : 'Missing'"
# )

In [ ]:
plt.plot(df_world['year'], 100*df_world['low_carbon_energy_per_capita']/df_world['energy_per_capita'])
plt.show()

# Plot 3
- ...

In [ ]:
drop_na_rows_importantCols = ['iso_code', 'population', 'gdp']
df_countries = df.dropna(subset = drop_na_rows_importantCols)
df_countries = df_countries[df_countries['year']>='1965']
df_countries

In [ ]:
df_countries['Low Carbon Energy Consumption (%)'] = 100*df_countries['low_carbon_energy_per_capita']/df_countries['energy_per_capita']

In [ ]:
alt.Chart(df_countries).mark_line().encode(
    x="year:T",
    y=alt.Y('Low Carbon Energy Consumption (%)'),
    tooltip=['country', 'year', 'energy_per_capita'],
    color=alt.Color("country:N").legend(None)
    # color=alt.Color('gdp_per_capita', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
)

In [ ]:
df_countries

In [ ]:
# df_countries2 = df_countries[df_countries['year']=='2018-01-01']
# alt.Chart(df_countries2).mark_bar().encode(
#     alt.X("Low Carbon Energy Consumption (%):Q").bin(maxbins=20).scale(domain=[1, 10]),
#     alt.Y('count()'),
#     alt.Color("Low Carbon Energy Consumption (%):Q").bin(maxbins=20).scale(scheme='pinkyellowgreen')
# )

In [ ]:
# def categorize_gdp(gdp):
#     if gdp < 5000:
#         return "Very Low GDP"
#     elif gdp < 15000:
#         return "Low GDP"
#     elif gdp < 30000:
#         return "Middle GDP"
#     elif gdp < 60000:
#         return "High GDP"
#     else:
#         return "Very High GDP"

# df_countries["gdp_category"] = df_countries["gdp_per_capita"].apply(categorize_gdp)

In [ ]:
df_countries.describe()

In [ ]:
# alt.Chart(df_countries).mark_line().encode(
#     x="year:T",
#     y=alt.Y('Low Carbon Energy Consumption (%)'),
#     tooltip=['country', 'year', 'energy_per_capita'],
#     color=alt.Color("gdp_category:N")#.legend(None)
#     # color=alt.Color('gdp_per_capita', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
# )

In [ ]:
# 1) pick the most recent non-missing gdp_per_capita per country
latest_gdp = (
    df_countries.dropna(subset=["gdp_per_capita"])
      .sort_values(["country", "year"])
      .groupby("country", as_index=False)
      .tail(1)[["country", "year", "gdp_per_capita"]]
      .rename(columns={"year": "latest_year", "gdp_per_capita": "latest_gdp_per_capita"})
)

# 2) create 5 categories based on latest_gdp_per_capita (quintiles)
latest_gdp["gdp_category"] = pd.qcut(
    latest_gdp["latest_gdp_per_capita"],
    q=5,
    labels=["Very Low", "Low", "Middle", "High", "Very High"]
)

# 3) merge back so every year gets the same category for that country
df_countries = df_countries.merge(latest_gdp[["country", "latest_year", "latest_gdp_per_capita", "gdp_category"]],
              on="country", how="left")

# df[["country", "year", "gdp_per_capita", "latest_year", "latest_gdp_per_capita", "gdp_category"]].head()
df_countries

In [ ]:
df_countries.gdp_category.unique()

In [ ]:
base = alt.Chart(df_countries).mark_circle().encode(
    x="year:T",
    y=alt.Y('Low Carbon Energy Consumption (%)').scale(domain=(0,100)),
    tooltip=['country', 'year', 'energy_per_capita'],
    color=alt.Color("gdp_category:N", title='GDP per Capita Category').sort(['Very Low', 'Low', 'Middle', 'High', 'Very High'])#.legend(None)
    # color=alt.Color('gdp_per_capita', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
).properties(
    title='Proportion of Energy from Low Carbon Sources\nCategorized by GDP per Capita'
)

chart = alt.vconcat()
for cat in ['Low', 'Middle', 'High', 'Very High']:
    chart |= base.transform_filter(alt.datum.gdp_category == cat)
chart

In [ ]:
alt.Chart(df_countries).mark_circle().encode(
    x="year:T",
    y=alt.Y('Low Carbon Energy Consumption (%)').scale(domain=(0,100)),
    tooltip=['country', 'year', 'energy_per_capita'],
    color=alt.Color("gdp_category:N", title='Category').sort(['Low', 'Middle', 'High', 'Very High']),#.legend(None)
    column=alt.Column('gdp_category:N', title='GDP per Capita Category').sort(['Low', 'Middle', 'High', 'Very High'])
    # color=alt.Color('gdp_per_capita', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
)

In [ ]:
df_countries = df_countries[df_countries['gdp_category'] != 'Very Low']

In [ ]:
alt.Chart(df_countries).mark_circle().encode(
    x="year:T",
    y=alt.Y('Low Carbon Energy Consumption (%)').scale(domain=(0,100)),
    tooltip=['country', 'year', 'energy_per_capita'],
    color=alt.Color("gdp_category:N", title='Category').sort(['Low', 'Middle', 'High', 'Very High']),#.legend(None)
    column=alt.Column('gdp_category:N', title='GDP per Capita Category').sort(['Low', 'Middle', 'High', 'Very High'])
    # color=alt.Color('gdp_per_capita', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
).configure_axis(
    labelFontSize=14,
    titleFontSize=16
).configure_header(
    titleFontSize=20,
    labelFontSize=16
).configure_legend(
    titleFontSize=20,
    labelFontSize=16
)

In [ ]:
# base = alt.Chart(df_countries).mark_circle().encode(
#     x="year:T",
#     y=alt.Y('Low Carbon Energy Consumption (%)').scale(domain=(0,100)),
#     tooltip=['country', 'year', 'energy_per_capita'],
#     color=alt.Color("gdp_category:N", title='Category').sort(['Low', 'Middle', 'High', 'Very High']),#.legend(None)
#     column=alt.Column('gdp_category:N', title='GDP per Capita Category').sort(['Low', 'Middle', 'High', 'Very High'])
#     # color=alt.Color('gdp_per_capita', title="Low Carbon Energy (%)").scale(scheme="yelloworangered", reverse=False),
# )

# trend = base.transform_regression(
#     "year:T", 'Low Carbon Energy Consumption (%)'
# ).mark_line(color='black')

# base + trend

# Plot 3
- most recent
- 

In [ ]:
df_countries_2018 = df_countries[df_countries['year'] == '2018-01-01']
df_countries_2018

In [ ]:


# trend = base.transform_regression(
#     'energy_per_capita:Q', 'low_carbon_share_energy'
# ).mark_line(color='black')


scatter3 = alt.Chart(df_countries_2018).mark_circle(size=100).encode(
    # x=alt.X('gdp_per_capita:Q', title='GDP per Capita',scale=alt.Scale(type='log')),
    # x=alt.X('gdp_per_capita:Q', title='GDP per Capita'),
    x=alt.X('energy_per_capita:Q', title='Energy Consumption per Capita (kWh/person)'),
    y=alt.Y('low_carbon_share_energy', title='Energy Consumed from Low Carbon Sources (%)'),
    # y=alt.Y('energy_per_capita:Q', title='Energy Demand per Capita (kWh/person)'),
    color=alt.Color("gdp_category:N", title='GDP Category').sort(['Low', 'Middle', 'High', 'Very High']),
    tooltip=['country', 'gdp_per_capita', 'energy_per_gdp', 'low_carbon_share_energy']
).properties(
    title='Low Carbon Energy Consumption in 2018',
    width=400,
    height=400
).configure_axis(
    labelFontSize=14,
    titleFontSize=16
).configure_header(
    titleFontSize=20,
    labelFontSize=16
).configure_legend(
    titleFontSize=20,
    labelFontSize=16
).configure_title(
    fontSize=20
)

# scatter3 + trend

scatter3

In [ ]:
df_countries.latest_year.unique()